# Load the datasets and the two models

In [ ]:
import joblib
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, confusion_matrix, r2_score,
                             ConfusionMatrixDisplay, mean_absolute_error)

from neuralns import mnn

DATASET_FOLDER = "../../data/datasets"
TUNER_RESULTS_FOLDER = "../../data/tuner_results"

print("Loading datasets...")
data = mnn.read_datasets(DATASET_FOLDER)
print("Loading models...")
model_with_t = joblib.load(f"{TUNER_RESULTS_FOLDER}/with/refit_best_model.mod")
model_wout_t = joblib.load(f"{TUNER_RESULTS_FOLDER}/wout/refit_best_model.mod")
print("Done")

# Report on the two models

In [ ]:
def report_model(model, title=None):
    """Report on a specific model's accuracy."""
    Y_reg_pred_test, Y_cls_pred_test = model.predict(data["test"])
    Y_reg_pred_hold, Y_cls_pred_hold = model.predict(data["holdout"])

    test_accuracy = accuracy_score(
        data['test'][model.transformer.y_cls_var], Y_cls_pred_test)
    hold_accuracy = accuracy_score(
        data['holdout'][model.transformer.y_cls_var], Y_cls_pred_hold)
    print(f"Accuracy (test): {test_accuracy:.3f}")
    print(f"Accuracy (hold): {hold_accuracy:.3f}")

    fig, ((ax4, ax3), (ax2, ax1)) = plt.subplots(
        2, 2, figsize=(8, 5), constrained_layout=True)
    fig.suptitle(title)
    ax1.plot(model.history.history['clsout_accuracy'], label="Training set")
    ax1.plot(model.history.history['val_clsout_accuracy'],
             label="Validation set")
    ax1.axhline(test_accuracy, label="Test set", color="C3")
    ax1.axhline(hold_accuracy, label="Holdout curves", color="C4")
    ax1.legend(loc="lower right")
    ax1.set_ylabel("Classification accuracy")

    test_mae = mean_absolute_error(
        data['test'][model.transformer.y_reg_vars], Y_reg_pred_test)
    hold_mae = mean_absolute_error(
        data['holdout'][model.transformer.y_reg_vars], Y_reg_pred_hold)

    print(f"MAE (test): {test_mae:.3f}")
    print(f"MAE (hold): {hold_mae:.3f}")

    ax2.plot(model.history.history['regout_mean_absolute_error'],
             label="Training set")
    ax2.plot(model.history.history['val_regout_mean_absolute_error'],
             label="Validation set")
    ax2.axhline(test_mae, label="Test set", color="C3")
    ax2.axhline(hold_mae, label="Holdout curves", color="C4")
    ax2.set_ylabel("Regression mean absolute error")

    ax3.plot(model.history.history['loss'], label='Training set')
    ax3.plot(model.history.history['val_loss'], label='Validation set')
    ax3.set_ylabel("Loss")

    if 'learning_rate' in model.history.history:
        ax4.plot(model.history.history['learning_rate'])
    ax4.set_ylabel("Learning rate")
    plt.show()

    cm_test = confusion_matrix(data["test"][model.transformer.y_cls_var],
                               Y_cls_pred_test,
                               labels=model.transformer.y_cls_encoder.classes_)
    cm_hold = confusion_matrix(data["holdout"][model.transformer.y_cls_var],
                               Y_cls_pred_hold,
                               labels=model.transformer.y_cls_encoder.classes_)
    cmn_test = cm_test.astype('float') / cm_test.sum(axis=1)[:, np.newaxis]
    cmn_hold = cm_hold.astype('float') / cm_hold.sum(axis=1)[:, np.newaxis]

    _, ((cm_ax1, cm_ax2), (cm_ax3, cm_ax4)) = plt.subplots(
        2, 2, figsize=(8, 5), constrained_layout=True)

    def make_cmdisplay(cm):
        return ConfusionMatrixDisplay(
            confusion_matrix=cm,
            display_labels=model.transformer.y_cls_encoder.classes_)

    disp1 = make_cmdisplay(cm_test)
    disp2 = make_cmdisplay(cmn_test)
    disp3 = make_cmdisplay(cm_hold)
    disp4 = make_cmdisplay(cmn_hold)
    for disp, ax, cmap, ax_title in zip(
            [disp1, disp2, disp3, disp4],
            [cm_ax1, cm_ax2, cm_ax3, cm_ax4],
            ["Blues", "Blues", "Greens", "Greens"],
            ["Test", "Test (normalized)", "Holdout", "Holdout (normalized)"]):
        disp.plot(cmap=cmap, ax=ax)
        ax.set_title(ax_title)

    plt.show()

    for i, var in enumerate(model.transformer.y_reg_vars):
        y_reg_pred_test = Y_reg_pred_test[var].values
        y_reg_true_test = data["test"][var].values
        y_reg_pred_hold = Y_reg_pred_hold[var].values
        y_reg_true_hold = data["holdout"][var].values

        r2_test = r2_score(y_reg_true_test, y_reg_pred_test)
        r2_hold = r2_score(y_reg_true_hold, y_reg_pred_hold)
        mae_test = mean_absolute_error(y_reg_true_test, y_reg_pred_test)
        mae_hold = mean_absolute_error(y_reg_true_hold, y_reg_pred_hold)
        var_title = (
            f"{var}:  test($R^2$={r2_test:.4g} -- MAE = {mae_test:.2g}) | "
            f"holdout($R^2$={r2_hold:.4g} -- MAE = {mae_hold:.2g})")

        plt.figure()
        plt.title(var_title)
        plt.plot(y_reg_true_test, y_reg_pred_test, ".",
                 color="k", alpha=0.25, label="Test")
        plt.plot(y_reg_true_hold, y_reg_pred_hold, ".",
                 color="r", alpha=0.25, label="Holdout")
        plt.xlabel("True")
        plt.ylabel("Predicted")
        plt.show()

report_model(model_wout_t, title="Model without time as input")
report_model(model_with_t, title="Model with time as input")

# Compare models on common output

In [ ]:
def compare_models(model1, model2, label1="Model 1", label2="Model 2"):
    Y_reg_pred_test1, Y_cls_pred_test1 = model1.predict(data["test"])
    Y_reg_pred_hold1, Y_cls_pred_hold1 = model1.predict(data["holdout"])
    Y_reg_pred_test2, Y_cls_pred_test2 = model2.predict(data["test"])
    Y_reg_pred_hold2, Y_cls_pred_hold2 = model2.predict(data["holdout"])

    cls_accuracy_test = accuracy_score(Y_cls_pred_test1, Y_cls_pred_test2)
    cls_accuracy_hold = accuracy_score(Y_cls_pred_hold1, Y_cls_pred_hold2)
    print("Classification accuracy between models (test)    :", cls_accuracy_test)
    print("Classification accuracy between models (holdout) :", cls_accuracy_hold)
    # hold_accuracy = accuracy_score(data['holdout'][model.transformer.y_cls_var], Y_cls_pred_hold)

    cm_test = confusion_matrix(Y_cls_pred_test1, Y_cls_pred_test2,
                               labels=model1.transformer.y_cls_encoder.classes_)
    cm_hold = confusion_matrix(Y_cls_pred_hold1, Y_cls_pred_hold2,
                               labels=model1.transformer.y_cls_encoder.classes_)
    cmn_test = cm_test.astype('float') / cm_test.sum(axis=1)[:, np.newaxis]
    cmn_hold = cm_hold.astype('float') / cm_hold.sum(axis=1)[:, np.newaxis]
    # # print("Confusion matrix (test):\n", cm_test)
    # # disp.plot(cmap="Blues")

    _, ((cm_ax1, cm_ax2), (cm_ax3, cm_ax4)) = plt.subplots(2, 2, figsize=(8, 5), constrained_layout=True)

    def make_cmdisplay(cm):
        return ConfusionMatrixDisplay(
            confusion_matrix=cm,
            display_labels=model1.transformer.y_cls_encoder.classes_)

    disp1 = make_cmdisplay(cm=cm_test)
    disp2 = make_cmdisplay(cm=cmn_test)
    disp3 = make_cmdisplay(cm=cm_hold)
    disp4 = make_cmdisplay(cm=cmn_hold)
    for disp, ax, cmap, ax_title in zip([disp1, disp2, disp3, disp4],
                                        [cm_ax1, cm_ax2, cm_ax3, cm_ax4],
                                        ["Blues", "Blues", "Greens", "Greens"],
                                        ["Test", "Test (normalized)", "Holdout", "Holdout (normalized)"]
                                       ):
        disp.plot(cmap=cmap, ax=ax)
        ax.set_title(ax_title)
        disp.ax_.set_xlabel(label2 + " prediction")
        disp.ax_.set_ylabel(label1 + " prediction")

    for var in model1.transformer.y_reg_vars:
        if var not in model2.transformer.y_reg_vars:
            continue
        y_reg_pred_test = Y_reg_pred_test1[var].values
        y_reg_true_test = Y_reg_pred_test2[var].values
        y_reg_pred_hold = Y_reg_pred_hold1[var].values
        y_reg_true_hold = Y_reg_pred_hold2[var].values

        r2_test = r2_score(y_reg_true_test, y_reg_pred_test)
        r2_hold = r2_score(y_reg_true_hold, y_reg_pred_hold)
        mae_test = mean_absolute_error(y_reg_true_test, y_reg_pred_test)
        mae_hold = mean_absolute_error(y_reg_true_hold, y_reg_pred_hold)
        var_title = f"{var}:  test($R^2$={r2_test:.4g} -- MAE = {mae_test:.2g}) | holdout($R^2$={r2_hold:.4g} -- MAE = {mae_hold:.2g})"

        plt.figure()
        plt.title(var_title)
        plt.plot(y_reg_true_test, y_reg_pred_test, ".", color="C3", alpha=0.25, label="Test")
        plt.plot(y_reg_true_hold, y_reg_pred_hold, ".", color="C4", alpha=0.25, label="Holdout")
        # plt.plot(yy, yy, "r--", alpha=0.7)
        plt.xlabel("Model without time")
        plt.ylabel("Model with time")
        plt.show()
    # disp1 = ConfusionMatrixDisplay(confusion_matrix=cm_test, display_labels=model.transformer.y_cls_encoder.classes_,)
    # disp2 = ConfusionMatrixDisplay(confusion_matrix=cmn_test, display_labels=model.transformer.y_cls_encoder.classes_)

compare_models(model_wout_t, model_with_t,
               label1="Model without time", label2="Model with time")